In [ ]:
from torch.nn import *
from IPython.display import clear_output
import torch
import numpy as np
import matplotlib.pyplot as plt
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class Add(Sequential):
    def forward(s, _):
        return sum(s(_)for s in s)
class Mul(Sequential):
    def forward(s, _):
        return s[0](_) * s[1](_)
class T(Sequential):
    def forward(s, x):
        return super().forward(x.mT).mT
class F(Module):
    def __init__(s, f, *_):
        super().__init__()
        s.f = f
        s._ = _
    def forward(s, _):
        return s.f(_, *s._)
A = torch.from_numpy(np.memmap('py.bin', mode='r'))
A = A.as_strided_((len(A)//512-1,1024),(512,1))
M = np.memmap('py.map', mode='r')

In [ ]:
class Attn(Linear):
    def __init__(s, i, r=4, g=1):
        super().__init__(i, i*g)
        s.r = r
    def forward(s, x):
        X = functional.pad(x,(0,0,(s.r-1)//2,s.r//2))
        X = X.as_strided_((*x.shape,s.r),(*X.stride(),X.stride(-2)))
        return ((super().forward(x).unflatten(-1,(-1,x.size(-1))) @ X).softmax(-1) @ X.mT).flatten(-2)
def SwiGLU(i, h):
    return Mul(
        Sequential(
            Linear(i, h),
            SiLU(),
        ),
        Linear(i, h),
    )
def Norm(i):
    return Add(
        LayerNorm(i, elementwise_affine=1, bias=0),
        T(Add(
            InstanceNorm1d(i, affine=1, track_running_stats=0),
            BatchNorm1d(i, affine=1, track_running_stats=0),
        )),
    )
def RoPE(x):
    _ = torch.arange(x.size(-2),dtype=x.dtype,device=x.device)[:,None]*torch.logspace(0,-4,x.size(-1)//2,dtype=x.dtype,device=x.device)
    return torch.cat((_.sin(), _.cos_()),1) + x
def Model(i, h, r, s, *_):
    return Sequential(
        Res(
            Sequential(
                Linear(i, i, 0),
                T(Conv1d(i, i, r, padding=r//2, groups=i, bias=0)),
            ) if isinstance(r, int) else Attn(i, int(r)),
            Norm(i),
            SwiGLU(i, h),
            Linear(h, i, 0),
        ),
        *(
            Res(
                Linear(i, h, 0),
                T(MaxPool1d(s)),
                Model(h, *_),
                Linear(h, h, 0),
                T(ConvTranspose1d(h, h, 3, 2, 1, 1, h, 0)),
                Linear(h, i, 0),
            ),
            Res(
                Sequential(
                    Linear(i, i, 0),
                    T(Conv1d(i, i, r, padding=r//2, groups=i, bias=0)),
                ) if isinstance(r, int) else Attn(i, int(r)),
                Norm(i),
                SwiGLU(i, h),
                Linear(h, i, 0),
            ),
        ) if _ else (),
    )
m = 16
m = Sequential(
    Embedding(92, m),
    F(RoPE),
    Model(m, m,7,2, m,7,2, m,7,2, m,7,2, m,7,2),
    Linear(m, 92),
).to('cuda',torch.bfloat16)
o = torch.optim.Adam(m.parameters(), 1/32)
L,TT,XX,XT = [],[],[],[]
for _ in range(500):
    for x in range(1):
        _ = 0
        for x in range(20):
            for x in range(20):
                t = A[torch.randint(len(A),[128])].to('cuda',torch.long)
                x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(92,[t.numel()//2],device='cuda'))
                y = m(x).mT
                l = functional.cross_entropy(y, t)
                (l*65536).backward()
                o.step()
                o.zero_grad()
            _ += l.data
        L.append(_[None])
        y = y.data.softmax(1)
        _ = x==t
        l = y[torch.arange(len(t),device='cuda')[:,None],t,torch.arange(t.size(1),device='cuda')]
        TT.append(l.mul(_).mean()[None])
        XT.append(l.mul_(_.logical_not_()).mean()[None])
        XX.append(y[torch.arange(len(t),device='cuda')[:,None],x,torch.arange(t.size(1),device='cuda')].mul_(_).mean()[None])
    plt.subplot(211).plot((L:=[torch.cat(L)])[0].to('cpu',torch.float))
    plt.subplot(212).plot((TT:=[torch.cat(TT)])[0].to('cpu',torch.float), label='TT')
    plt.plot((XX:=[torch.cat(XX)])[0].to('cpu',torch.float), label='XX')
    plt.plot((XT:=[torch.cat(XT)])[0] .to('cpu',torch.float), label='XT')
    plt.legend()
    clear_output(True)
    plt.show()
    print(M[x[0,448:576].cpu()].view('S128')[0])
    print(M[y[0,:,448:576].argmax(0).cpu()].view('S128')[0])